# Recolección de datos meteorológicos

## Fuente de datos

Los datos meteorológicos fueron obtenidos mediante la API histórica de **Open-Meteo**.

La consulta fue realizada utilizando coordenadas geográficas representativas de cada región de Chile, obteniendo valores meteorológicos diarios para el periodo comprendido entre enero de 2020 y julio de 2026.

Para cada región se utilizaron las siguientes variables:

- Temperatura media diaria.
- Temperatura máxima diaria.
- Temperatura mínima diaria.
- Precipitación acumulada diaria.

Los datos obtenidos fueron posteriormente asociados a su respectiva región para permitir la integración con el dataset de precios agrícolas.

---

## Estructura del dataset climático

Cada fila representa un día de observación meteorológica para una región determinada.

| Variable | Tipo | Descripción |
|-----------|------|-------------|
| `time` | Fecha | Fecha correspondiente al registro meteorológico diario. |
| `temperature_2m_mean` | Numérica | Temperatura promedio diaria a 2 metros sobre la superficie (°C). |
| `temperature_2m_max` | Numérica | Temperatura máxima registrada durante el día a 2 metros de altura (°C). |
| `temperature_2m_min` | Numérica | Temperatura mínima registrada durante el día a 2 metros de altura (°C). |
| `precipitation_sum` | Numérica | Precipitación acumulada durante el día (mm). |
| `region` | Categórica | Región de Chile asociada al punto geográfico utilizado para la consulta. |

---

## Consideraciones metodológicas

- La API entrega información meteorológica asociada a coordenadas específicas, por lo que se utilizaron puntos representativos cercanos a las capitales regionales.
- La variable `region` fue agregada manualmente durante la etapa de extracción, ya que la fuente meteorológica no entrega información administrativa.
- La frecuencia original de los datos es diaria.

In [1]:
import requests
import pandas as pd

In [2]:
time_periods = [ 
        ["2020-01-01","2021-01-01"],
        ["2021-01-02","2022-01-01"],
        ["2022-01-02","2023-01-01"],
        ["2023-01-02","2024-01-01"],
        ["2024-01-02","2025-01-01"],
        ["2025-01-02","2026-07-01"]
    ]

In [3]:
regions = {
    "Región de Arica y Parinacota": (-18.4783, -70.3126),
    "Región de Tarapacá": (-20.2141, -70.1520),
    "Región de Antofagasta": (-23.6509, -70.3975),
    "Región de Atacama": (-27.3668, -70.3323),
    "Región de Coquimbo": (-29.9027, -71.2519),
    "Región de Valparaíso": (-33.0472, -71.6127),
    "Región Metropolitana": (-33.4489, -70.6693),
    "Región del Libertador General Bernardo O'Higgins": (-34.1708, -70.7406),
    "Región del Maule": (-35.4264, -71.6554),
    "Región de Ñuble": (-36.6066, -72.1034),
    "Región del Biobío": (-36.8201, -73.0444),
    "Región de La Araucanía": (-38.7359, -72.5904),
    "Región de Los Ríos": (-39.8196, -73.2452),
    "Región de Los Lagos": (-41.4693, -72.9424),
    "Región de Aysén": (-45.5712, -72.0683),
    "Región de Magallanes y de la Antártica Chilena": (-53.1638, -70.9171)
}

## Parámetros de extracción

La extracción fue realizada iterando sobre:

1. Regiones geográficas de Chile usando sus respectivas latitudes y longitudes.
2. Periodos temporales definidos entre 2020 y 2026.

Para cada combinación región-periodo se realizó una solicitud independiente a la API.

In [4]:
def getMeteoData(start_date,end_date,region,coordinates):
    url = "https://archive-api.open-meteo.com/v1/archive"

    lat = coordinates[0]
    lon = coordinates[1]
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum"
        ],
        "timezone": "America/Santiago"
    }
    
    response = requests.get(url, params=params)
    
    print("Status code:", response.status_code)
    if response.status_code == 200:
        data = response.json()
    
        df = pd.DataFrame(data["daily"])
        df['region'] = region
        return df
    else:
        print(response.text)
        return None

In [5]:
dfs = {}

contador = 0

for region, coordinates in regions.items():
    for tp in time_periods:
        dfs[contador] = getMeteoData(tp[0], tp[1], region, coordinates)
        contador += 1

Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 2

## Revision de las request

In [6]:
len(time_periods)* len(regions)
len(dfs)

96

In [ ]:
dfs[0].head()
dfs[0].shape
dfs[0].isna().sum()

In [ ]:
dfs[1].head()
dfs[1].shape
dfs[1].isna().sum()

## Consolidación de datos

Los resultados individuales obtenidos para cada región y periodo fueron concatenados en un único dataset climático, manteniendo la trazabilidad mediante la columna `region`.

In [7]:
df_meteo_final = pd.concat(dfs.values(), ignore_index=True)

In [24]:
df_meteo_final

,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,region
0,2020-01-01,23.0,26.2,20.9,0.0,Región de Arica y Parinacota
1,2020-01-02,23.0,26.1,20.4,2.1,Región de Arica y Parinacota
2,2020-01-03,23.6,27.8,20.4,1.0,Región de Arica y Parinacota
3,2020-01-04,23.5,26.4,20.5,0.0,Región de Arica y Parinacota
4,2020-01-05,23.9,27.4,21.3,0.0,Región de Arica y Parinacota
...,...,...,...,...,...,...
37067,2023-12-28,6.7,10.2,4.2,1.7,Región de Magallanes y de la Antártica Chilena
37068,2023-12-29,8.7,11.1,4.7,2.3,Región de Magallanes y de la Antártica Chilena
37069,2023-12-30,10.8,15.1,7.7,0.0,Región de Magallanes y de la Antártica Chilena
37070,2023-12-31,10.7,13.6,8.3,1.0,Región de Magallanes y de la Antártica Chilena


In [ ]:
df_meteo_final.shape

In [18]:
df_meteo_final.to_csv("./Data/meteo.csv", index=False)